**Objetivo:** Carregar os dados brutos do SIH e CNES, filtrar internações por IAM (CID I21), realizar limpeza e integrar as duas bases.

**Outputs gerados:**
- `data/interim/sih_iam.parquet` — internações IAM limpas
- `data/interim/cnes_hospitais.parquet` — dados hospitalares consolidados
- `data/processed/base_modelagem.parquet` — base final (SIH + CNES)

## Configuração do ambiente

In [1]:
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path
import sys

In [2]:
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print(f"Raiz do projeto: {ROOT}")

Raiz do projeto: /home/carolina/Documents/TCC Documentos/TCC


In [3]:
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

#  Caminhos 
RAW_SIH     = Path(ROOT,'data/input/SIH')
RAW_CNES    = Path(ROOT,'data/input/CNES')
INTERIM     = Path(ROOT,'data/interim')
PROCESSED   = Path(ROOT,'data/processed')

INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

# CIDs de interesse: I21 e seus subcódigos (I21.0 a I21.9)
CID_IAM = ['I21', 'I210', 'I211', 'I212', 'I213', 'I214',
            'I219', 'I21.0', 'I21.1', 'I21.2', 'I21.3',
            'I21.4', 'I21.9']

# Separador dos CSVs do pysus (ajuste se necessário)
SEP = ';'

# Encoding padrão DATASUS
ENCODING = 'latin-1'

print('✓ Configurações carregadas')
print(f'  SIH  → {RAW_SIH}')
print(f'  CNES → {RAW_CNES}')

✓ Configurações carregadas
  SIH  → /home/carolina/Documents/TCC Documentos/TCC/data/input/SIH
  CNES → /home/carolina/Documents/TCC Documentos/TCC/data/input/CNES


In [4]:
arquivos_sih = sorted(glob.glob(str(RAW_SIH / '*.csv')))
print(f'Arquivos SIH encontrados: {len(arquivos_sih)}')
for f in arquivos_sih:
    print(f'  {Path(f).name}')

Arquivos SIH encontrados: 12
  rdsp2501.csv
  rdsp2502.csv
  rdsp2503.csv
  rdsp2504.csv
  rdsp2505.csv
  rdsp2506.csv
  rdsp2507.csv
  rdsp2508.csv
  rdsp2509.csv
  rdsp2510.csv
  rdsp2511.csv
  rdsp2512.csv


In [6]:
arquivos_sih

['/home/carolina/Documents/TCC Documentos/TCC/data/input/SIH/rdsp2501.csv',
 '/home/carolina/Documents/TCC Documentos/TCC/data/input/SIH/rdsp2502.csv',
 '/home/carolina/Documents/TCC Documentos/TCC/data/input/SIH/rdsp2503.csv',
 '/home/carolina/Documents/TCC Documentos/TCC/data/input/SIH/rdsp2504.csv',
 '/home/carolina/Documents/TCC Documentos/TCC/data/input/SIH/rdsp2505.csv',
 '/home/carolina/Documents/TCC Documentos/TCC/data/input/SIH/rdsp2506.csv',
 '/home/carolina/Documents/TCC Documentos/TCC/data/input/SIH/rdsp2507.csv',
 '/home/carolina/Documents/TCC Documentos/TCC/data/input/SIH/rdsp2508.csv',
 '/home/carolina/Documents/TCC Documentos/TCC/data/input/SIH/rdsp2509.csv',
 '/home/carolina/Documents/TCC Documentos/TCC/data/input/SIH/rdsp2510.csv',
 '/home/carolina/Documents/TCC Documentos/TCC/data/input/SIH/rdsp2511.csv',
 '/home/carolina/Documents/TCC Documentos/TCC/data/input/SIH/rdsp2512.csv']

In [5]:
# Inspeciona o primeiro arquivo para entender a estrutura
df_sample = pd.read_csv(arquivos_sih[0], sep=SEP, encoding=ENCODING, nrows=5)
print(f'Shape (amostra): {df_sample.shape}')
print(f'\nColunas ({len(df_sample.columns)}):')  
print(df_sample.columns.tolist())


Shape (amostra): (5, 1)

Colunas (1):
['UF_ZI,ANO_CMPT,MES_CMPT,ESPEC,CGC_HOSP,N_AIH,IDENT,CEP,MUNIC_RES,NASC,SEXO,UTI_MES_IN,UTI_MES_AN,UTI_MES_AL,UTI_MES_TO,MARCA_UTI,UTI_INT_IN,UTI_INT_AN,UTI_INT_AL,UTI_INT_TO,DIAR_ACOM,QT_DIARIAS,PROC_SOLIC,PROC_REA,VAL_SH,VAL_SP,VAL_SADT,VAL_RN,VAL_ACOMP,VAL_ORTP,VAL_SANGUE,VAL_SADTSR,VAL_TRANSP,VAL_OBSANG,VAL_PED1AC,VAL_TOT,VAL_UTI,US_TOT,DT_INTER,DT_SAIDA,DIAG_PRINC,DIAG_SECUN,COBRANCA,NATUREZA,NAT_JUR,GESTAO,RUBRICA,IND_VDRL,MUNIC_MOV,COD_IDADE,IDADE,DIAS_PERM,MORTE,NACIONAL,NUM_PROC,CAR_INT,TOT_PT_SP,CPF_AUT,HOMONIMO,NUM_FILHOS,INSTRU,CID_NOTIF,CONTRACEP1,CONTRACEP2,GESTRISCO,INSC_PN,SEQ_AIH5,CBOR,CNAER,VINCPREV,GESTOR_COD,GESTOR_TP,GESTOR_CPF,GESTOR_DT,CNES,CNPJ_MANT,INFEHOSP,CID_ASSO,CID_MORTE,COMPLEX,FINANC,FAEC_TP,REGCT,RACA_COR,ETNIA,SEQUENCIA,REMESSA,AUD_JUST,SIS_JUST,VAL_SH_FED,VAL_SP_FED,VAL_SH_GES,VAL_SP_GES,VAL_UCI,MARCA_UCI,DIAGSEC1,DIAGSEC2,DIAGSEC3,DIAGSEC4,DIAGSEC5,DIAGSEC6,DIAGSEC7,DIAGSEC8,DIAGSEC9,TPDISEC1,TPDISEC2,TPDISEC3,TP